# molecule-gpt — experiments

These are the three experiments required by the assignment: the effect of **temperature**, the effect of **prompt length and context**, and the effect of an **architectural parameter** (`n_layer`).

It reuses the checkpoint trained by the `molecule-gpt` kernel, mounted as an input.

**Before running:** Settings -> Internet **ON**, Accelerator **GPU T4**, and add the `molecule-gpt` kernel under Data -> Notebook Output.

In [ ]:
import os
%cd /kaggle/working
if os.path.exists('molecule-gpt'):
    !git -C molecule-gpt pull
else:
    !git clone https://github.com/hi-z-k/molecule-gpt
%cd /kaggle/working/molecule-gpt

In [ ]:
!pip install -q rdkit 2>&1 | tail -1
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## Locating the trained checkpoint and training data

The previous kernel's output is mounted read-only under `/kaggle/input`. The checkpoint is copied into the repo layout the scripts expect. `train.csv` is needed to compute novelty: it is reused from the mounted output if present, otherwise regenerated with the identical `--limit` so the split matches.

In [ ]:
import glob, os, shutil

cands = glob.glob('/kaggle/input/**/best.pt', recursive=True)
if not cands:
    raise SystemExit("No best.pt found under /kaggle/input. Add the `molecule-gpt` kernel's output "
                     "under Data -> Notebook Output, and make sure the training kernel has finished.")
chosen = next((c for c in cands if 'smiles' in c), cands[0]) if len(cands) > 1 else cands[0]
print('candidates:')
for c in cands:
    print('  ', c)
print('chosen:', chosen)

os.makedirs('checkpoints/smiles', exist_ok=True)
shutil.copy(chosen, 'checkpoints/smiles/best.pt')

trains = glob.glob('/kaggle/input/**/processed/train.csv', recursive=True)
if trains:
    src_dir = os.path.dirname(trains[0])
    os.makedirs('data/processed', exist_ok=True)
    for name in ('train.csv', 'val.csv', 'meta.json'):
        shutil.copy(os.path.join(src_dir, name), 'data/processed/' + name)
    with open(trains[0]) as f:
        n_lines = sum(1 for _ in f)
    print('train.csv from', trains[0], '-', n_lines, 'lines')
    NEED_PREP = False
else:
    print('train.csv was not in the mounted output; it will be regenerated')
    NEED_PREP = True

import torch
ck = torch.load('checkpoints/smiles/best.pt', map_location='cpu', weights_only=False)
print('checkpoint step:', ck['step'], ' val_loss:', ck['val_loss'])

In [ ]:
# --limit 600000 must match the training kernel so the novelty reference set is the same molecules.
if NEED_PREP:
    !python scripts/prepare_data.py --out_dir data/processed --limit 600000 2>&1 | tail -8
else:
    print('reusing train/val split from the mounted kernel output')

## 1. Effect of temperature

Temperature rescales the logits before the softmax: low values concentrate probability on the argmax, high values flatten the distribution. The expectation is that validity falls and uniqueness rises as temperature grows, and that T=0 (greedy) produces one identical string every time, so uniqueness collapses.

In [ ]:
!python scripts/experiments.py temperature --n 2000 --batch_size 256 --out_dir results/experiments 2>&1 | tail -20

In [ ]:
from IPython.display import Image, display
display(Image('results/experiments/temperature.png'))

## 2. Effect of prompt length and context

Two measurements:

- (a) mean cross-entropy at each position index over the validation set, which shows how much easier the next character is to predict once more of the molecule is visible;
- (b) completions of real validation-molecule prefixes of increasing length, scored for validity, which tests whether a longer committed prefix constrains the model into finishing a legal molecule.

In [ ]:
!python scripts/experiments.py context --n 1000 --batch_size 256 --out_dir results/experiments 2>&1 | tail -20

In [ ]:
display(Image('results/experiments/context.png'))

## 3. Effect of architectural parameters

This trains models from scratch varying **one** architectural parameter at a time, with every
other hyperparameter and the data subset held fixed, so any difference in validation loss is
attributable to that parameter alone. Three axes are swept:

| axis | values | what it changes |
| --- | --- | --- |
| `n_layer` | 2, 4, 8 | depth — how many times the residual stream is refined |
| `n_head` | 2, 4, 8 | how many attention subspaces per block, at fixed total width |
| `n_embd` | 64, 128, 256 | model width — the size of every token's representation |

The baseline is the shape of the main model (`n_head` 8, `n_embd` 256, `block_size` 128).
Note that `n_head` is swept at *fixed* `n_embd`, so more heads means a smaller per-head
dimension `d_k = n_embd / n_head` — this isolates how attention is partitioned, not capacity.

These are short runs (1200 steps on a 200k-molecule subset, each capped at 15 minutes) and are
therefore a controlled comparison between settings, **not** converged models. The headline
numbers come from the main training run.

In [ ]:
# no --values: each axis uses its own defaults, since one shared list cannot
# suit all three (e.g. n_embd must stay divisible by n_head)
!python scripts/experiments.py ablation --axis all \
    --ab_steps 1200 --ab_rows 200000 --ab_batch_size 256 --ab_block_size 128 \
    --ab_base_head 8 --ab_base_embd 256 --ab_eval_interval 300 --ab_eval_samples 500 \
    --ab_max_minutes 15 --out_dir results/experiments 2>&1 | tail -40

In [ ]:
for axis in ('n_layer', 'n_head', 'n_embd'):
    path = f'results/experiments/ablation_{axis}.png'
    if os.path.exists(path):
        print(axis)
        display(Image(path))

## Summary

`results/experiments/summary.md` holds every table in markdown, ready to paste into the technical report.

In [ ]:
print(open('results/experiments/summary.md').read())

In [ ]:
import shutil, os
out = '/kaggle/working/experiment-outputs'
shutil.rmtree(out, ignore_errors=True)
shutil.copytree('results/experiments', out)
shutil.make_archive('/kaggle/working/molecule-gpt-experiments', 'zip', out)
print(sorted(os.listdir(out)))